In [1]:
import os
import random
from datetime import datetime, timedelta
import pandas as pd
from sqlalchemy import create_engine, text

In [2]:
# ============================================================================
# 1. DATABASE CONNECTION CONFIGURATION
# ============================================================================
DB_USER = os.getenv("DB_USER", "postgres")
DB_PASSWORD = os.getenv("DB_PASSWORD", "Dikshu1998")
DB_HOST = os.getenv("DB_HOST", "localhost")
DB_PORT = os.getenv("DB_PORT", "5432")
DB_NAME = os.getenv("DB_NAME", "recycling_db")

DATABASE_URL = f"postgresql+psycopg2://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
engine = create_engine(DATABASE_URL, echo=False)

In [3]:
# ============================================================================
# 2. DDL EXECUTION (SCHEMA CREATION)
# ============================================================================
DDL_SCHEMA = """
DROP TABLE IF EXISTS outbound_dispatch CASCADE;
DROP TABLE IF EXISTS bag_production CASCADE;
DROP TABLE IF EXISTS qa_lab_tests CASCADE;
DROP TABLE IF EXISTS extrusion_compounding CASCADE;
DROP TABLE IF EXISTS wash_batches CASCADE;
DROP TABLE IF EXISTS weighbridge_intake CASCADE;

CREATE TABLE weighbridge_intake (
    ticket_id INT PRIMARY KEY,
    supplier_id VARCHAR(30) NOT NULL,
    hauler_id VARCHAR(30) NOT NULL,
    arrival_time TIMESTAMP NOT NULL,
    polymer_type VARCHAR(10) NOT NULL,
    gross_weight_kg NUMERIC(10,2) NOT NULL,
    tare_weight_kg NUMERIC(10,2) NOT NULL,
    net_weight_kg NUMERIC(10,2) GENERATED ALWAYS AS (gross_weight_kg - tare_weight_kg) STORED,
    sample_contamination_pct NUMERIC(5,2) NOT NULL,
    price_per_tonne NUMERIC(10,2) NOT NULL
);

CREATE TABLE wash_batches (
    batch_id INT PRIMARY KEY,
    ticket_id INT NOT NULL REFERENCES weighbridge_intake(ticket_id),
    line_id VARCHAR(10) NOT NULL,
    start_time TIMESTAMP NOT NULL,
    end_time TIMESTAMP NOT NULL,
    feed_mass_kg NUMERIC(10,2) NOT NULL,
    clean_flake_output_kg NUMERIC(10,2) NOT NULL,
    reject_waste_kg NUMERIC(10,2) NOT NULL,
    water_consumption_m3 NUMERIC(10,2) NOT NULL,
    energy_kwh NUMERIC(10,2) NOT NULL
);

CREATE TABLE extrusion_compounding (
    extrusion_id INT PRIMARY KEY,
    batch_id INT NOT NULL REFERENCES wash_batches(batch_id),
    extruder_id VARCHAR(10) NOT NULL,
    start_time TIMESTAMP NOT NULL,
    end_time TIMESTAMP NOT NULL,
    flake_input_kg NUMERIC(10,2) NOT NULL,
    virgin_blend_ratio_pct NUMERIC(5,2) NOT NULL,
    virgin_input_kg NUMERIC(10,2) NOT NULL,
    masterbatch_kg NUMERIC(10,2) NOT NULL,
    melt_temp_c NUMERIC(5,1) NOT NULL,
    screen_filter_pressure_bar NUMERIC(5,1) NOT NULL,
    pellet_yield_kg NUMERIC(10,2) NOT NULL,
    extruder_kwh NUMERIC(10,2) NOT NULL
);

CREATE TABLE qa_lab_tests (
    test_id INT PRIMARY KEY,
    extrusion_id INT NOT NULL REFERENCES extrusion_compounding(extrusion_id),
    test_timestamp TIMESTAMP NOT NULL,
    melt_flow_index NUMERIC(5,2) NOT NULL,
    density_g_cm3 NUMERIC(5,4) NOT NULL,
    moisture_pct NUMERIC(5,3) NOT NULL,
    tensile_break_strength_mpa NUMERIC(5,2) NOT NULL,
    qa_status VARCHAR(15) NOT NULL CHECK (qa_status IN ('PASSED', 'REJECTED', 'DOWNGRADED'))
);

CREATE TABLE bag_production (
    work_order_id INT PRIMARY KEY,
    extrusion_id INT NOT NULL REFERENCES extrusion_compounding(extrusion_id),
    machine_id VARCHAR(10) NOT NULL,
    sku_id VARCHAR(40) NOT NULL,
    start_time TIMESTAMP NOT NULL,
    end_time TIMESTAMP NOT NULL,
    pellet_input_kg NUMERIC(10,2) NOT NULL,
    units_produced INT NOT NULL,
    total_roll_mass_kg NUMERIC(10,2) NOT NULL,
    scrap_kg NUMERIC(10,2) NOT NULL,
    downtime_minutes INT NOT NULL,
    downtime_reason VARCHAR(50) NOT NULL
);

CREATE TABLE outbound_dispatch (
    dispatch_id INT PRIMARY KEY,
    work_order_id INT NOT NULL REFERENCES bag_production(work_order_id),
    customer_id VARCHAR(40) NOT NULL,
    delivery_note_ref VARCHAR(30) NOT NULL,
    ship_date TIMESTAMP NOT NULL,
    units_shipped INT NOT NULL,
    net_mass_shipped_kg NUMERIC(10,2) NOT NULL,
    certified_pcr_pct NUMERIC(5,2) NOT NULL,
    avoided_co2e_kg NUMERIC(10,2) NOT NULL,
    invoice_value_eur NUMERIC(10,2) NOT NULL
);
"""

def init_database(conn):
    print("Executing DDL to initialize database tables...")
    for statement in DDL_SCHEMA.strip().split(";"):
        stmt_clean = statement.strip()
        if stmt_clean:
            conn.execute(text(stmt_clean))
    print("Database schema successfully created.")

In [4]:
# ============================================================================
# 3. SYNTHETIC DATA GENERATOR
# ============================================================================
def generate_plant_data():
    print("Generating synthetic production records...")
    random.seed(42)

    # 1. Weighbridge Intake
    suppliers = ['SUP-PANDA-WASTE', 'SUP-BEAUPARC-IRE', 'SUP-BIFFA-LOGISTICS', 'SUP-TESCO-DC', 'SUP-AGRI-RECYCLE']
    haulers = ['FREIGHT-EXPRESS', 'TRANS-IRISH-LOG', 'GREEN-LINE-LOG', 'DIRECT-HAUL']
    polymers = ['LDPE', 'HDPE', 'PET']
    weighbridge_data = []
    base_date = datetime(2026, 9, 1, 6, 30)

    for ticket_id in range(101, 161):
        supplier = random.choice(suppliers)
        polymer = random.choice(polymers)
        tare = round(random.uniform(13800, 14500), 2)
        net = round(random.uniform(11000, 17000), 2)
        gross = tare + net

        contamination = (
            round(random.uniform(6.5, 14.8), 2)
            if supplier in ['SUP-BEAUPARC-IRE', 'SUP-AGRI-RECYCLE']
            else round(random.uniform(1.2, 4.9), 2)
        )

        if polymer == 'PET':
            price_per_tonne = 520.00
        elif polymer == 'HDPE':
            price_per_tonne = 440.00
        else:
            price_per_tonne = 415.00

        arrival = base_date + timedelta(hours=(ticket_id - 101) * 1.5, minutes=random.randint(0, 40))

        weighbridge_data.append({
            'ticket_id': ticket_id,
            'supplier_id': supplier,
            'hauler_id': random.choice(haulers),
            'arrival_time': arrival,
            'polymer_type': polymer,
            'gross_weight_kg': gross,
            'tare_weight_kg': tare,
            'sample_contamination_pct': contamination,
            'price_per_tonne': price_per_tonne
        })
    df_weighbridge = pd.DataFrame(weighbridge_data)

    # 2. Wash Batches
    wash_data = []
    batch_counter = 501

    for _, ticket in df_weighbridge.iterrows():
        net_weight = ticket['gross_weight_kg'] - ticket['tare_weight_kg']
        num_batches = 2 if net_weight > 13500 else 1
        split_mass = round(net_weight / num_batches, 2)

        for i in range(num_batches):
            line = 'LINE_1' if ticket['polymer_type'] == 'LDPE' else 'LINE_2'
            start_time = ticket['arrival_time'] + timedelta(hours=1 + (i * 3.5), minutes=random.randint(5, 25))
            end_time = start_time + timedelta(hours=3, minutes=random.randint(15, 45))

            shrinkage = (ticket['sample_contamination_pct'] / 100.0) + random.uniform(0.06, 0.10)
            clean_output = round(split_mass * (1.0 - shrinkage), 2)
            rejects = round((split_mass - clean_output) * random.uniform(0.75, 0.88), 2)

            water_m3 = round((split_mass / 1000.0) * random.uniform(1.8, 2.6), 2)
            kwh_rate = 0.32 if line == 'LINE_1' else 0.42
            energy_kwh = round(clean_output * random.uniform(kwh_rate * 0.95, kwh_rate * 1.25), 2)

            wash_data.append({
                'batch_id': batch_counter,
                'ticket_id': ticket['ticket_id'],
                'line_id': line,
                'start_time': start_time,
                'end_time': end_time,
                'feed_mass_kg': split_mass,
                'clean_flake_output_kg': clean_output,
                'reject_waste_kg': rejects,
                'water_consumption_m3': water_m3,
                'energy_kwh': energy_kwh
            })
            batch_counter += 1

    df_wash = pd.DataFrame(wash_data)

    # 3. Compounding & Extrusion
    extrusion_data = []
    ext_counter = 701

    for _, wash in df_wash.iterrows():
        flake_input = wash['clean_flake_output_kg']
        virgin_ratio = random.choice([0.05, 0.10, 0.15, 0.20])
        virgin_kg = round(flake_input * (virgin_ratio / (1.0 - virgin_ratio)), 2)
        masterbatch_kg = round((flake_input + virgin_kg) * 0.02, 2)

        total_charge = flake_input + virgin_kg + masterbatch_kg
        pellet_yield = round(total_charge * (1.0 - random.uniform(0.015, 0.030)), 2)

        start_time = wash['end_time'] + timedelta(hours=1, minutes=random.randint(10, 30))
        end_time = start_time + timedelta(hours=4, minutes=random.randint(0, 30))

        base_pressure = 135.0 if wash['reject_waste_kg'] < 600 else 185.0
        screen_pressure = round(base_pressure + random.uniform(5.0, 45.0), 1)

        extrusion_data.append({
            'extrusion_id': ext_counter,
            'batch_id': wash['batch_id'],
            'extruder_id': random.choice(['EXT-01', 'EXT-02']),
            'start_time': start_time,
            'end_time': end_time,
            'flake_input_kg': flake_input,
            'virgin_blend_ratio_pct': round(virgin_ratio * 100, 2),
            'virgin_input_kg': virgin_kg,
            'masterbatch_kg': masterbatch_kg,
            'melt_temp_c': round(random.uniform(195.0, 218.0), 1),
            'screen_filter_pressure_bar': screen_pressure,
            'pellet_yield_kg': pellet_yield,
            'extruder_kwh': round(pellet_yield * random.uniform(0.40, 0.52), 2)
        })
        ext_counter += 1

    df_extrusion = pd.DataFrame(extrusion_data)

    # 4. QA Lab Tests
    qa_data = []
    qa_counter = 1001

    for _, ext in df_extrusion.iterrows():
        test_time = ext['end_time'] + timedelta(minutes=random.randint(15, 45))
        mfi = round(random.uniform(0.65, 1.45), 2)
        density = round(random.uniform(0.9210, 0.9280), 4)
        moisture = round(random.uniform(0.018, 0.075), 3)
        tensile = round(random.uniform(16.5, 26.0), 2)

        if moisture > 0.055 or mfi > 1.35 or tensile < 17.5:
            status = 'REJECTED' if moisture > 0.07 else 'DOWNGRADED'
        else:
            status = 'PASSED'

        qa_data.append({
            'test_id': qa_counter,
            'extrusion_id': ext['extrusion_id'],
            'test_timestamp': test_time,
            'melt_flow_index': mfi,
            'density_g_cm3': density,
            'moisture_pct': moisture,
            'tensile_break_strength_mpa': tensile,
            'qa_status': status
        })
        qa_counter += 1

    df_qa = pd.DataFrame(qa_data)

    # 5. Bag Production (Conversion)
    production_data = []
    prod_counter = 9001
    sku_catalog = {
        'BAG-REFUSE-85L': {'unit_mass_kg': 0.18, 'scrap_base': 0.035},
        'BAG-WHEELIE-240L': {'unit_mass_kg': 0.38, 'scrap_base': 0.040},
        'BAG-RUBBLE-HEAVY': {'unit_mass_kg': 0.65, 'scrap_base': 0.055}
    }
    downtime_causes = ['SEAL_FAILURE', 'BUBBLE_BURST', 'CHANGEOVER', 'KNIFE_BLUNT', 'HEATER_FAULT']
    downtime_weights = [0.42, 0.32, 0.14, 0.08, 0.04]

    for _, ext in df_extrusion.iterrows():
        qa_row = df_qa[df_qa['extrusion_id'] == ext['extrusion_id']].iloc[0]
        if qa_row['qa_status'] == 'REJECTED':
            continue

        sku_key = random.choice(list(sku_catalog.keys()))
        sku_spec = sku_catalog[sku_key]

        start_time = ext['end_time'] + timedelta(hours=random.randint(2, 6))
        end_time = start_time + timedelta(hours=random.randint(5, 8))

        pellet_input = ext['pellet_yield_kg']
        scrap_rate = sku_spec['scrap_base'] + (0.04 if qa_row['qa_status'] == 'DOWNGRADED' else 0.01)
        scrap_kg = round(pellet_input * scrap_rate, 2)
        roll_mass = round(pellet_input - scrap_kg, 2)

        production_data.append({
            'work_order_id': prod_counter,
            'extrusion_id': ext['extrusion_id'],
            'machine_id': random.choice(['BAG_M01', 'BAG_M02', 'BAG_M03']),
            'sku_id': sku_key,
            'start_time': start_time,
            'end_time': end_time,
            'pellet_input_kg': pellet_input,
            'units_produced': int(roll_mass / sku_spec['unit_mass_kg']),
            'total_roll_mass_kg': roll_mass,
            'scrap_kg': scrap_kg,
            'downtime_minutes': random.randint(15, 60) if qa_row['qa_status'] == 'PASSED' else random.randint(70, 165),
            'downtime_reason': random.choices(downtime_causes, weights=downtime_weights)[0]
        })
        prod_counter += 1

    df_production = pd.DataFrame(production_data)

    # 6. Outbound Dispatch
    dispatch_data = []
    dispatch_counter = 4001
    customers = ['CUST-DUBLIN-COUNCIL', 'CUST-TESCO-IE', 'CUST-BUNZL-CLEAN', 'CUST-MONAGHAN-COUNTY', 'CUST-BWG-FOODS']

    for _, prod in df_production.iterrows():
        ext_parent = df_extrusion[df_extrusion['extrusion_id'] == prod['extrusion_id']].iloc[0]
        total_recipe = ext_parent['flake_input_kg'] + ext_parent['virgin_input_kg'] + ext_parent['masterbatch_kg']
        certified_pcr = round((ext_parent['flake_input_kg'] / total_recipe) * 100.0, 2)

        ship_date = prod['end_time'] + timedelta(days=random.randint(1, 4), hours=random.randint(1, 5))
        mass_shipped = prod['total_roll_mass_kg']
        avoided_co2e = round(mass_shipped * (certified_pcr / 100.0) * 1.85, 2)
        unit_price = 0.35 if '85L' in prod['sku_id'] else (0.75 if '240L' in prod['sku_id'] else 1.20)

        dispatch_data.append({
            'dispatch_id': dispatch_counter,
            'work_order_id': prod['work_order_id'],
            'customer_id': random.choice(customers),
            'delivery_note_ref': f"DN-2026-{dispatch_counter}",
            'ship_date': ship_date,
            'units_shipped': prod['units_produced'],
            'net_mass_shipped_kg': mass_shipped,
            'certified_pcr_pct': certified_pcr,
            'avoided_co2e_kg': avoided_co2e,
            'invoice_value_eur': round(prod['units_produced'] * unit_price, 2)
        })
        dispatch_counter += 1

    df_dispatch = pd.DataFrame(dispatch_data)

    return df_weighbridge, df_wash, df_extrusion, df_qa, df_production, df_dispatch

In [5]:
# ============================================================================
# 4. INGESTION PIPELINE (ORDERED INSERTS)
# ============================================================================
def load_data_to_postgres():
    with engine.begin() as conn:
        init_database(conn)

    dfs = generate_plant_data()
    table_names = [
        'weighbridge_intake',
        'wash_batches',
        'extrusion_compounding',
        'qa_lab_tests',
        'bag_production',
        'outbound_dispatch'
    ]

    print("\nStarting batch insertions into PostgreSQL...")
    for df, table_name in zip(dfs, table_names):
        # Insert using append mode and multi-row execution for high speed
        df.to_sql(
            name=table_name,
            con=engine,
            if_exists='append',
            index=False,
            method='multi',
            chunksize=1000
        )
        print(f"Successfully loaded {len(df):>4} records into '{table_name}'")

    print("\nAuditing database table counts:")
    with engine.connect() as conn:
        for table_name in table_names:
            result = conn.execute(text(f"SELECT COUNT(*) FROM {table_name}")).scalar()
            print(f"  - {table_name:<25}: {result} rows verified")

if __name__ == "__main__":
    load_data_to_postgres()

Executing DDL to initialize database tables...
Database schema successfully created.
Generating synthetic production records...

Starting batch insertions into PostgreSQL...
Successfully loaded   60 records into 'weighbridge_intake'
Successfully loaded   92 records into 'wash_batches'
Successfully loaded   92 records into 'extrusion_compounding'
Successfully loaded   92 records into 'qa_lab_tests'
Successfully loaded   83 records into 'bag_production'
Successfully loaded   83 records into 'outbound_dispatch'

Auditing database table counts:
  - weighbridge_intake       : 60 rows verified
  - wash_batches             : 92 rows verified
  - extrusion_compounding    : 92 rows verified
  - qa_lab_tests             : 92 rows verified
  - bag_production           : 83 rows verified
  - outbound_dispatch        : 83 rows verified
